## Running scPROTEIN stage2 on our dataset

In [ ]:
import argparse
import random
import numpy as np 
import scipy.sparse as sp
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch_geometric.deprecation")

os.chdir('/Users/apple/Desktop/SCP_Projects/SCP_Microglia')

from sklearn import metrics
from sklearn.metrics import silhouette_score,adjusted_rand_score,normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix
from scprotein import *

data_dir = '/Volumes/SU700/scpMS_data/scpMS_microglia/scpMS/'
fig_dir = 'SCP_Microglia/figures/1_4_batch_correction/'
results_dir = 'SCP_Microglia/results/1_4_batch_correction/'

In [2]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

parser = argparse.ArgumentParser()
parser.add_argument("--stage1", type=bool, default=False, help='if scPROTEIN starts from stage1')
parser.add_argument("--learning_rate", type=float, default=1e-3, help='learning rate')
parser.add_argument("--num_hidden", type=int, default=400, help='hidden dimension') 
parser.add_argument("--num_proj_hidden", type=int, default=256, help='dimension of projection head')
parser.add_argument("--activation", type=str, default='prelu', help='activation function') 
parser.add_argument("--num_layers", type=int, default=2, help='num of GCN layers')
parser.add_argument("--num_protos", type=int, default=2, help='num of prototypes')
parser.add_argument("--num_changed_edges", type=int, default=50, help='num of added/removed edges')
parser.add_argument("--topology_denoising", type=bool, default=False, help='if scPROTEIN uses topology denoising')
parser.add_argument("--drop_edge_rate_1", type=float, default=0.2, help='dropedge rate for view1')
parser.add_argument("--drop_edge_rate_2", type=float, default=0.4, help='dropedge rate for view2')
parser.add_argument("--drop_feature_rate_1", type=float, default=0.4, help='mask_feature rate for view1')
parser.add_argument("--drop_feature_rate_2", type=float, default=0.2, help='mask_feature rate for view2')
parser.add_argument("--alpha", type=float, default=0.05, help='balance factor')
parser.add_argument("--tau", type=float, default=0.4, help='temperature coefficient')
parser.add_argument("--weight_decay", type=float, default=0.00001, help='weight_decay')
parser.add_argument("--num_epochs", type=int, default=200, help='Number of epochs.')
parser.add_argument("--seed", type=int, default=39788, help='Random seed.') 
parser.add_argument("--threshold", type=float, default=0.15, help='threshold of graph construct')
parser.add_argument("--feature_preprocess", type=bool, default=True, help='feature preprocess')
args =parser.parse_known_args()[0]   
setup_seed(args.seed)
activation = nn.PReLU() if args.activation == 'prelu' else F.relu

In [ ]:
# load log-normalized matrix
features = np.load('scPROTEIN/data/adata_matrix.npy')

In [ ]:
# normalize the data by median and mean, this step is important for training stability
features = np.where(features == 0, np.nan, features)

features = features - np.nanmedian(features, axis=0)
features = features - np.nanmean(features, axis=1, keepdims=True)

features = np.where(np.isnan(features), 0, features)

In [ ]:
 device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
data = graph_generation(features, args.threshold, args.feature_preprocess).to(device)
torch.cuda.empty_cache()
encoder = Encoder(data.num_features, args.num_hidden, activation, k=args.num_layers).to(device)
model = Model(encoder, args.num_hidden, args.num_proj_hidden, args.tau).to(device)
scPROTEIN = scPROTEIN_learning(model,device, data, args.drop_feature_rate_1,args.drop_feature_rate_2,args.drop_edge_rate_1,args.drop_edge_rate_2,
                 args.learning_rate, args.weight_decay, args.num_protos, args.topology_denoising, args.num_epochs, args.alpha, args.num_changed_edges,args.seed)

### Model training

In [6]:
scPROTEIN.train()

(T) | Epoch=001, loss=8.3433 
(T) | Epoch=002, loss=7.5951 
(T) | Epoch=003, loss=7.2689 
(T) | Epoch=004, loss=7.1382 
(T) | Epoch=005, loss=7.0545 
(T) | Epoch=006, loss=6.9917 
(T) | Epoch=007, loss=6.9465 
(T) | Epoch=008, loss=6.9294 
(T) | Epoch=009, loss=6.9079 
(T) | Epoch=010, loss=6.8841 
(T) | Epoch=011, loss=6.8667 
(T) | Epoch=012, loss=6.8432 
(T) | Epoch=013, loss=6.8333 
(T) | Epoch=014, loss=6.8209 
(T) | Epoch=015, loss=6.7913 
(T) | Epoch=016, loss=6.7958 
(T) | Epoch=017, loss=6.7737 
(T) | Epoch=018, loss=6.7614 
(T) | Epoch=019, loss=6.7407 
(T) | Epoch=020, loss=6.7504 
(T) | Epoch=021, loss=6.7235 
(T) | Epoch=022, loss=6.7161 
(T) | Epoch=023, loss=6.7124 
(T) | Epoch=024, loss=6.6912 
(T) | Epoch=025, loss=6.6958 
(T) | Epoch=026, loss=6.6858 
(T) | Epoch=027, loss=6.6767 
(T) | Epoch=028, loss=6.6683 
(T) | Epoch=029, loss=6.6479 
(T) | Epoch=030, loss=6.6549 
(T) | Epoch=031, loss=6.6478 
(T) | Epoch=032, loss=6.6385 
(T) | Epoch=033, loss=6.6316 
(T) | Epoc

### Generate the cell embedding

In [7]:
embedding = scPROTEIN.embedding_generation()
embedding.shape

(3068, 400)

In [ ]:
np.save(results_dir + 'scprotein_embedding.npy', embedding)